## 具有平方误差的梯度下降
我们想找到神经网络的权重。 让我们开始思考目标。 网络需要使预测尽可能接近实际值。 为了衡量这一点，我们使用一个度量来衡量预测的错误程度。 通用度量是平方误差（SSE）的总和：
$$E = \frac{1}{2}\Sigma_\mu\Sigma_j[y^\mu_j - \hat{y}^\mu_j]^2$$
其中$\hat{y}$是预测，$y$是真实值，公式将所有输出单位$j$的总和与所有数据点$\mu$的另一个总和相乘。

首先，$j$的内部总和。 该变量$j$代表网络的输出单元。 因此，这个内部总和就是说对于每个输出单元，从网络$\hat{y}$找到真实值$y$和预测值之间的差，然后对差进行平方，然后对所有这些平方求和。

然后，$\mu$上的另一个和是所有数据点的和。 因此，对于每个数据点，您可以计算每个输出单位的平方差的内部和。 然后对每个数据点的平方差求和。 这样就为您提供了所有数据点的所有输出预测的总体误差。

---

出于以下几个原因，SSE是一个不错的选择。 平方确保误差始终为正，并且较大的误差要比较小的误差受到更多的惩罚。 而且，它使数学很好，始终是加分项。

请记住，神经网络的输出（预测）取决于权重
$$\hat{y}^\mu_j = f(\Sigma_i\Omega_{ij}x^\mu_i)$$
因此，误差取决于权重
$$E = \frac{1}{2}\Sigma_\mu\Sigma_j[y^\mu_j - f(\Sigma_i w_{ij}x^\mu_i)]^2$$
我们希望网络的预测误差尽可能小，权重是我们可以用来实现此目标的旋钮。 我们的目标是找到最小化平方误差$E$的权重$w_{ij}$。 要通过神经网络执行此操作，通常会使用**梯度下降**。

梯度只是泛化为具有多个变量的函数的导数。 我们可以使用微积分在误差函数的任何点上找到梯度，这取决于输入权重。

**注意事项**<br>
由于权重会随梯度的变化而变化，因此权重可能会落在误差很低但不是最低的地方。 这些斑点称为局部极小值。 如果权重使用错误的值初始化，则梯度下降可能导致权重达到局部最小值

## 对于仅一个输出单元的情况
从之前我们可以看到一个权重更新可以计算为：
$$\Delta w_i = \eta\delta x_i$$
误差项$\delta$为
$$\delta = (y - \hat{y})f'(h) = (y - \hat{y})f'(\Sigma w_i x_i)$$
请记住，在上面的等式$(y - \hat{y})$是输出误差，$f'(h)$指激活函数$f(h)$的导数，我们称该导数为输出梯度。

我将使用S形作为激活函数$f(h)$

In [13]:
import numpy as np

# 定义Sigmoid函数进行激活
def sigmoid(x):
    return 1/(1+np.exp(-x))

# S型函数的导数
def sigmoid_prime(x):
    return sigmoid(x) * (1 - sigmoid(x))

# 输入数据
x = np.array([0.1, 0.3])
# 目标
y = 0.2
# 输入输出权重
weights = np.array([-0.8, 0.5])

# 权重阶跃方程中的学习率
learnrate = 0.5

# TODO: 计算节点的输入和权重的线性组合 (f(h)和f'(h)中的h)
h = x[0]*weights[0] + x[1]*weights[1]
# 或者 h = np.dot(x, weights)

# TODO: 计算神经网络的输出 (y-hat)
nn_output = sigmoid(h)

# TODO: 计算神经网络的误差 (y - y-hat)
error = y - nn_output

# TODO: 输出梯度 (f'(h))
output_grad = sigmoid_prime(h)

# TODO: 计算误差项
error_term = error * output_grad

# TODO: 梯度下降步骤，计算权重变化
del_w = [ learnrate * error_term * x[0],
          learnrate * error_term * x[1]]
# 或者 del_w = learnrate * error_term * x

print('神经网络输出：')
print(nn_output)
print('误差量：')
print(error)
print('权重变化：')
print(del_w)

神经网络输出：
0.5174928576663897
误差量：
-0.31749285766638974
权重变化：
[-0.003963803079006883, -0.011891409237020648]


您已经看到了如何为单个更新实现该功能，但是我们如何转换该代码以计算许多权重更新，以便我们的网络能够学习？

## 实施梯度下降
举例来说，我将使用梯度下降来训练有关研究生入学数据的网络。 该数据集具有三个输入功能：GRE得分，GPA和本科学校的排名（从1到4）。 排名第一的机构威望最高，排名第四的机构最低。<br>
此处的目标是根据这些功能预测学生是否会被录取进入研究生课程。 为此，我们将使用具有一个输出层和一个单元的网络。 我们将使用S形函数来激活输出单元。

### 数据整理

In [18]:
import numpy as np
import pandas as pd

admissions = pd.read_csv('binary.csv')

您可能认为会有三个输入单元，但实际上我们需要首先转换数据。等级功能是绝对的，数字不编码任何相对值。等级2并不是等级1的两倍，等级3则不是等级2的1.5。<br>所以，我们需要使用虚拟变量对等级进行编码，将数据分为四个用1或0编码的新列。等级为1的行在等级1的虚拟列中为1，在所有其他列中为零。 等级2的行在等级2的虚拟列中为1，在所有其他列中为0，以此类推。

In [19]:
# 为等级设置虚拟变量
data = pd.concat([admissions, pd.get_dummies(admissions['rank'], prefix='rank')], axis=1)
data = data.drop('rank', axis=1)

我们还需要对GRE和GPA数据进行标准化，这意味着要缩放这些值，使其平均值为零，标准偏差为1。这是必需的，因为S型函数会压缩非常小的输入和非常大的输入。 实际大小输入的梯度为零，这意味着梯度下降步骤也将变为零。 <br>由于GRE和GPA值相当大，因此我们必须非常小心如何初始化权重，否则梯度下降步将消失并且网络将无法训练。 <br>所以，如果我们对数据进行标准化，则可以轻松地初始化权重。

In [20]:
# 标准化
for field in ['gre', 'gpa']:
    mean, std = data[field].mean(), data[field].std()
    data.loc[:,field] = (data[field]-mean)/std

把数据分成训练集和测试集

In [21]:
# 随机抽取10％的数据进行测试
np.random.seed(42)
sample = np.random.choice(data.index, size=int(len(data)*0.9), replace=False)
data, test_data = data.iloc[sample], data.drop(sample)

# 分为特征和目标
features, targets = data.drop('admit', axis=1), data['admit']
features_test, targets_test = test_data.drop('admit', axis=1), test_data['admit']

### 均方误差
我们将对此处计算错误的方式进行一些小的更改。 代替SSE，我们将使用平方误差（MSE）的均值。 <br>
现在我们正在使用大量数据，总结所有权重步骤可能会导致非常大的更新，从而使梯度下降有所不同。 为了弥补这一点，您需要使用很小的学习率。 <br>所以，我们可以将数据中的记录数除以mmm取平均值。 这样，无论我们使用多少数据，我们的学习率通常将在0.01到0.001的范围内。 <br>
然后，我们可以使用MSE（如下所示）来计算梯度，结果与以前相同，只是取平均值而不是求和。

**这是使用梯度下降更新权重的通用算法：**
- 将权重步长设置为零: $\Delta w_i = 0$
- 对于训练数据中的每个记录: 
 - 通过神经网络前向传递，计算输出: $\hat{y} = f(\Sigma_i w_i x_i)$
 - 计算输出单元的误差项: $\delta = (y - \hat{y}) * f'(\Sigma_i w_i x_i)$
 - 更新权重步骤: $\Delta w_i = \Delta w_i + \delta x_i$
- 更新权重: $w_i = w_i + \eta\Delta w_i/m$
 - $\eta$是学习率，$m$是记录数。
 - 在这里，我们对权重步骤进行平均，以帮助减少训练数据中的任何较大差异。
- 重复进行迭代。

您也可以更新每条记录的权重，而不是遍历所有记录后平均权重的步长。

请记住，我们使用S型信号来激活功能<br>
$$f(h) = 1/(1+e^{-h})$$<br>
Sigmoid函数的梯度是<br>
$$f'(h) = f(h)(1-f(h))$$<br>
$h$是输出单元的输入<br>
$$h = \Sigma_i w_i x_i$$

## 用NumPy实施
首先，您需要初始化权重。 我们希望它们很小，以使S形输入在接近0的线性区域内，而不会在高端和低端被压缩。 随机初始化它们也很重要，以使它们都具有不同的起始值并且发散，破坏对称性。 因此，我们将从以0为中心的正态分布中初始化权重。比例尺的一个好的值是$\frac{1}{\sqrt n}$，其中$n$是输入单位的数量。这将使S型输入保持低电平，以增加输入单元的数量。
```
weights = np.random.normal(scale=1/n_features**.5, size=n_features)
```
NumPy提供了一个函数`np.dot()`来计算两个数组的点积，从而方便地为我们计算$h$。 点积将两个数组按元素相乘，将数组1中的第一个元素与数组2中的第一个元素相乘，依此类推。 然后，对每个乘积求和。
```
# 输入到输出层
output_in = np.dot(weights, inputs)
```
最后，我们可以通过以`weight+=...`递增$\Delta w_i$和$w_i$来更新

## 效率提示
您可以保存一些计算，因为我们在这里使用了S形。 对于S型函数，$f'(h) = f(h)(1-f(h))。$这意味着一旦计算出输出单元的激活函数 $f(h)$ ，就可以使用它来计算误差梯度的梯度。

## 编程练习

在下面，您将实现梯度下降，并在准入数据上训练网络。 您的目标是训练网络，直到在训练集上达到最小均方误差（MSE）为止。 您需要实现：
- 网络输出: `output`.
- 输出误差: `error`.
- 误差项: `error_term`.
- 更新权重步骤: `del_w +=`.
- 更新权重: `weights +=`.

MSE将打印出正确预测的入学分数以及测试集的准确性。

In [24]:
# 定义Sigmoid函数进行激活
def sigmoid(x):
    return 1/(1+np.exp(-x))

# 我们没有像上一课中那样提供sigmoid_prime函数

# 用于同一种子使调试更容易
np.random.seed(42)

n_records, n_features = features.shape
last_loss = None

# 初始化权重
weights = np.random.normal(scale=1 / n_features**.5, size=n_features)

# 神经网络超参数
epochs = 1000
learnrate = 0.5

for e in range(epochs):
    del_w = np.zeros(weights.shape)
    for x, y in zip(features.values, targets):
        # 遍历所有记录，x是输入，y是目标

        # 激活输出单元
        #   注意，我们在这里将输入和权重相乘
        #   而不是将h存储为单独的变量
        output = sigmoid(np.dot(x, weights))

        # 误差，目标减去网络输出
        error = y - output

        # 误差项
        #   注意，我们在这里计算f'(h)，而不是定义单独的sigmoid_prime函数。
        #   这只是使速度更快，因为我们可以重复使用存储在输出变量中的sigmoid函数的结果
        error_term = error * output * (1 - output)

        # 梯度下降步骤，误差乘以梯度乘以输入
        del_w += error_term * x

    # 在此处更新权重。 学习率乘以权数变化＃除以平均记录数
    weights += learnrate * del_w / n_records

    # 在训练集上打印均方误差
    if e % (epochs / 10) == 0:
        out = sigmoid(np.dot(features, weights))
        loss = np.mean((out - targets) ** 2)
        if last_loss and last_loss < loss:
            print("训练损失: ", loss, "  WARNING - 损失增加")
        else:
            print("训练损失: ", loss)
        last_loss = loss


# 计算测试数据的准确性
tes_out = sigmoid(np.dot(features_test, weights))
predictions = tes_out > 0.5
accuracy = np.mean(predictions == targets_test)
print("预测精度:  {:.3f}".format(accuracy))

训练损失:  0.2627609384996635
训练损失:  0.20928619409324875
训练损失:  0.20084292908073426
训练损失:  0.19862156475527873
训练损失:  0.1977985139668603
训练损失:  0.19742577912189863
训练损失:  0.1972350774624106
训练损失:  0.1971294562509248
训练损失:  0.19706766341315082
训练损失:  0.19703005801777368
预测精度:  0.725
